# DOT Bronze Ingestion

This notebook ingests the DOT **Regional Distribution of Overnight Travelers** PDFs (DS_05) from Google Drive into Bronze Delta tables.

Each PDF is read directly from Google Drive through the existing Unity Catalog connection, its table rows are extracted **exactly as printed** (all values kept as text), reconciliation checks are run, and results are logged to the shared ingestion control tables created by `00_ingestion_setup`.

**Bronze outputs** (a rerun replaces only the selected years; other years are untouched):

| Table | One row = |
| --- | --- |
| `dot_ds_05_<year>` (one table per year) | one printed table row (region, province/HUC, or sub-row) |
| `dot_ds_05_report_text` (all years) | one non-table line (titles, "as of" date, footnotes such as the Boracay jetty port note), with `source_year` |
| `dot_ds_05_pdf` (all years) | one original PDF stored as binary, for traceability, with `source_year` |

No geography mapping, deduplication, type conversion, or missing-value handling happens here. That belongs in Silver.

In [0]:
%pip install pdfplumber==0.11.9

In [0]:
from pyspark.sql import functions as F
import uuid, io
from datetime import datetime

# ── Google Drive connection (existing Unity Catalog connection) ──
GOOGLE_DRIVE_CONNECTION = "capstone_gdrive"

# ── Unity Catalog targets ────────────────────────────────────────
CATALOG = "tuklas_dev"
BRONZE_SCHEMA = "02_bronze"
INGESTION_SCHEMA = "01_ingestion"
NOTEBOOK_NAME = "03_ingest_dot"
SOURCE_SYSTEM = "DOT"
DATASET_ID = "DS_05"

run_id = f"dot_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"

# ── DOT source folders ───────────────────────────────────────────
# DOT root folder (for reference). The yearly PDFs sit inside the Regional Travelers subfolder.
DOT_ROOT_FOLDER = "https://drive.google.com/drive/folders/12pSL9t9MsD4wy7Jj0L_Qadb5UsfxZJ3m"
# Regional Travelers folder: listed below and used to read each file's exact Drive filename.
DOT_FOLDER = "https://drive.google.com/drive/folders/1dlmJFnL7FZgd3KXnioxcbrbmI5bS_ovy"

# ── One entry per yearly PDF. "file" is filled automatically from the folder listing below. ──
SOURCE_FILES = {
    "2019": {"url": "https://drive.google.com/file/d/1lb7xOE-JP61BOK443DtGeHsD6niu6KMK/view", "file": None},
    "2020": {"url": "https://drive.google.com/file/d/1AsitfPtsb4EEKMFUWskyoIcwStRHqhk5/view", "file": None},
    "2021": {"url": "https://drive.google.com/file/d/1-pW315Sj4g6BtgdRcNsqEsZ4lqOJ4aFi/view", "file": None},
    "2022": {"url": "https://drive.google.com/file/d/1vBIbCyjGbLD2hChtd68XRZmdWe8onrfE/view", "file": None},
    "2023": {"url": "https://drive.google.com/file/d/1Jo_vNbvlBhEnWvLeGzjdsFLSpLMbok0z/view", "file": None},
    "2024": {"url": "https://drive.google.com/file/d/10d4dEmJRHaUS-4RoObkD4Rj2NAsEkmm4/view", "file": None},
}

# Years to process in this run: "all", or a list like ["2019"] for the pilot
YEARS_TO_RUN = "all"

# ── Bronze tables: one data table per year + one combined table each for report text and PDFs ──
def tbl_rows(year): return f"dot_ds_05_{year}"
TBL_TEXT = "dot_ds_05_report_text"
TBL_PDF  = "dot_ds_05_pdf"

def bronze_table(name): return f"`{CATALOG}`.`{BRONZE_SCHEMA}`.`{name}`"
def ingestion_table(name): return f"`{CATALOG}`.`{INGESTION_SCHEMA}`.`{name}`"

METADATA_COLS = ["_source_dataset_id", "_source_file", "_source_sheet", "_source_path", "_source_system", "_ingested_at"]

def add_bronze_metadata(df, source_file, source_path):
    return (df
        .withColumn("_source_dataset_id", F.lit(DATASET_ID))
        .withColumn("_source_file", F.lit(source_file))
        .withColumn("_source_sheet", F.lit(None).cast("string"))
        .withColumn("_source_path", F.lit(source_path))
        .withColumn("_source_system", F.lit(SOURCE_SYSTEM))
        .withColumn("_ingested_at", F.current_timestamp()))

ingestion_summary = []

def record_ingestion(source_file, bronze_tbl, row_count, col_count, status, notes="", source_path=None):
    """Append to the in-memory summary and MERGE one row into file_manifest (same pattern as 01_ingest_psa)."""
    ingestion_summary.append({"source_dataset_id": DATASET_ID, "source_file": source_file, "bronze_table": bronze_tbl,
                              "row_count": row_count, "column_count": col_count, "status": status, "notes": notes})
    row = spark.createDataFrame([(run_id, NOTEBOOK_NAME, DATASET_ID, SOURCE_SYSTEM, source_file, None, source_path,
                                  bronze_tbl, row_count, col_count, status, notes)],
        schema="run_id STRING, notebook_name STRING, source_dataset_id STRING, source_system STRING, source_file STRING, "
               "source_sheet STRING, source_path STRING, destination_table STRING, row_count BIGINT, column_count BIGINT, "
               "status STRING, notes STRING")
    row = row.withColumn("discovered_at", F.current_timestamp()).withColumn("ingested_at", F.current_timestamp())
    row.createOrReplaceTempView("_manifest_row")
    spark.sql(f"""
    MERGE INTO {ingestion_table('file_manifest')} AS target
    USING _manifest_row AS source
    ON target.run_id = source.run_id
       AND target.source_dataset_id = source.source_dataset_id
       AND target.source_file = source.source_file
       AND COALESCE(target.source_sheet, '') = COALESCE(source.source_sheet, '')
       AND target.destination_table = source.destination_table
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    """)

def log_ingestion_error(source_file, source_path, destination_table, error_type, error_message, notes=""):
    row = spark.createDataFrame([(run_id, NOTEBOOK_NAME, DATASET_ID, SOURCE_SYSTEM, source_file, None, source_path,
                                  destination_table, error_type, str(error_message)[:4000], None, notes)],
        schema="run_id STRING, notebook_name STRING, source_dataset_id STRING, source_system STRING, source_file STRING, "
               "source_sheet STRING, source_path STRING, destination_table STRING, error_type STRING, "
               "error_message STRING, error_timestamp TIMESTAMP, notes STRING")
    row = row.withColumn("error_timestamp", F.current_timestamp())
    row.write.format("delta").mode("append").saveAsTable(ingestion_table("ingestion_errors"))

def write_table(df, tbl):
    """Each year has its own tables, so a rerun simply overwrites that year's table (no duplicates)."""
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(bronze_table(tbl)))
    
def write_year_part(df, tbl, year):
    """Combined table: replace only this year's rows; other years are left untouched."""
    if spark.catalog.tableExists(bronze_table(tbl)):
        (df.write.format("delta").mode("overwrite")
           .option("replaceWhere", f"source_year = '{year}'")
           .saveAsTable(bronze_table(tbl)))
    else:
        df.write.format("delta").mode("overwrite").saveAsTable(bronze_table(tbl))

years = list(SOURCE_FILES) if YEARS_TO_RUN == "all" else YEARS_TO_RUN
print(f"Run ID: {run_id}\nYears : {years}")

## PDF Extraction Functions

Tested on the actual DOT files for 2019–2024. Two things these functions handle:

* **Hidden blank characters:** some yearly PDFs (e.g., 2020–2022, 2024) contain blank characters printed on top of digits, which split numbers like `458,951` into `4 58,951`. They are removed before characters are grouped into words.
* **Indentation:** each printed row's indent level (0 = region or GRAND TOTAL, 1 = province/HUC, 2 = sub-row such as a municipality) and its parent label are recorded **as printed**. Note: Boracay is printed at level 1, the same level as provinces, in every year, so its parent is the region, not Aklan.

Checks run per file:

* every row: Foreign + Overseas Filipinos + Domestic = Total
* sum of region totals = GRAND TOTAL
* exactly 3 indent levels and 1 GRAND TOTAL row
* region total vs. its level-1 rows (informational only, recorded in notes)

In [0]:
import io, re
from collections import defaultdict
import pdfplumber

VALUE_RE = re.compile(r"^(-|\d{1,3}(,\d{3})*)$")
REGION_RE = re.compile(r"^(NCR|CAR|Region\b|MIMAROPA)")
YEAR_RE = re.compile(r"JANUARY\s*-\s*DECEMBER\s+(\d{4})", re.I)
ASOF_RE = re.compile(r"\bas\s+of\s+(.+)$", re.I)

def _to_int(x):
    return 0 if x == "-" else int(x.replace(",", ""))

def parse_dot_pdf(pdf_bytes):
    """Extract every printed line from a DOT Regional Travelers PDF.

    Returns (data_rows, text_lines, info). Values are kept exactly as printed (text).
    Overlapping blank characters (present in some years and splitting numbers like
    '4 58,951') are removed before grouping characters into words.
    """
    data_rows, text_lines = [], []
    with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
        n_pages = len(pdf.pages)
        line_no = 0
        for page_no, page in enumerate(pdf.pages, start=1):
            clean = page.filter(lambda o: not (o.get("object_type") == "char" and o["text"].isspace()))
            lines = defaultdict(list)
            for w in clean.extract_words(x_tolerance=1.5, y_tolerance=2):
                lines[round(w["top"])].append(w)
            for top in sorted(lines):
                words = sorted(lines[top], key=lambda w: w["x0"])
                toks = [w["text"] for w in words]
                line_no += 1
                if len(toks) >= 5 and all(VALUE_RE.match(t) for t in toks[-4:]):
                    data_rows.append({
                        "page_number": page_no, "pdf_line_number": line_no,
                        "label_x0": round(words[0]["x0"], 1),
                        "area_label": " ".join(toks[:-4]),
                        "foreign_travelers": toks[-4], "overseas_filipinos": toks[-3],
                        "domestic_travelers": toks[-2], "total_travelers": toks[-1],
                    })
                else:
                    text_lines.append({"page_number": page_no, "pdf_line_number": line_no,
                                       "line_text": " ".join(toks)})

    # Indentation levels: cluster label x positions (3 levels expected: region / province-HUC / sub-row)
    xs = sorted({round(r["label_x0"]) for r in data_rows})
    clusters = []
    for x in xs:
        if not clusters or x - clusters[-1][-1] > 4:
            clusters.append([x])
        else:
            clusters[-1].append(x)
    def level_of(x):
        x = round(x)
        return next(i for i, c in enumerate(clusters) if c[0] - 1 <= x <= c[-1] + 1)

    stack = {}
    for r in data_rows:
        lvl = level_of(r["label_x0"])
        r["indent_level"] = lvl
        r["parent_label"] = stack.get(lvl - 1) if lvl > 0 else None
        stack[lvl] = r["area_label"]
        for k in [k for k in stack if k > lvl]:
            del stack[k]

    first_page_text = " ".join(t["line_text"] for t in text_lines if t["page_number"] == 1)
    year = YEAR_RE.search(first_page_text)
    as_of = next((ASOF_RE.search(t["line_text"]).group(1) for t in text_lines
                  if t["page_number"] == 1 and ASOF_RE.search(t["line_text"])), None)
    info = {"pages": n_pages, "report_year": year.group(1) if year else None,
            "report_as_of": as_of, "indent_clusters": clusters,
            "report_title": text_lines[0]["line_text"] if text_lines else None}
    return data_rows, text_lines, info

def check_extraction(data_rows, info):
    """Reconciliation checks. Returns (critical_errors, notes)."""
    errors, notes = [], []
    if len(info["indent_clusters"]) != 3:
        errors.append(f"Expected 3 indent levels, found {len(info['indent_clusters'])}")
    bad_rows = [r["area_label"] for r in data_rows
                if sum(_to_int(r[c]) for c in ("foreign_travelers", "overseas_filipinos", "domestic_travelers"))
                != _to_int(r["total_travelers"])]
    if bad_rows:
        errors.append(f"{len(bad_rows)} rows where Foreign+Overseas+Domestic != Total (e.g., {bad_rows[:3]})")
    grand = [r for r in data_rows if r["indent_level"] == 0 and "TOTAL" in r["area_label"].upper()]
    regions = [i for i, r in enumerate(data_rows) if r["indent_level"] == 0 and REGION_RE.match(r["area_label"])]
    if len(grand) != 1:
        errors.append(f"Expected 1 GRAND TOTAL row, found {len(grand)}")
    else:
        s = sum(_to_int(data_rows[i]["total_travelers"]) for i in regions)
        if s != _to_int(grand[0]["total_travelers"]):
            errors.append(f"Sum of regions ({s:,}) != GRAND TOTAL ({_to_int(grand[0]['total_travelers']):,})")
    # Region total vs. its level-1 rows (informational: Boracay is printed as its own level-1 row)
    bounds = regions + [len(data_rows)]
    for a, b in zip(bounds, bounds[1:]):
        if a == bounds[-2] and b == len(data_rows):
            b = next((i for i in range(a + 1, len(data_rows)) if data_rows[i]["indent_level"] == 0), b)
        kids = [r for r in data_rows[a + 1:b] if r["indent_level"] == 1]
        diff = sum(_to_int(r["total_travelers"]) for r in kids) - _to_int(data_rows[a]["total_travelers"])
        if diff:
            same = [r["area_label"] for r in kids if _to_int(r["total_travelers"]) == diff]
            notes.append(f"{data_rows[a]['area_label']}: level-1 rows exceed region total by {diff:,}"
                         + (f" (same value as: {', '.join(same)})" if same else ""))
    return errors, notes


In [0]:
spark.sql(f"""
INSERT INTO {ingestion_table('ingestion_run_log')}
(run_id, notebook_name, source_system, source_folder, started_at, completed_at,
 files_discovered, files_ingested, files_skipped, files_failed, tables_created, status, notes)
VALUES (:run_id, :nb, :sys, :folder, current_timestamp(), NULL, 0, 0, 0, 0, 0, 'STARTED', 'Notebook execution started')
""", args={"run_id": run_id, "nb": NOTEBOOK_NAME, "sys": SOURCE_SYSTEM, "folder": DOT_FOLDER})
print(f"✓ STARTED record inserted for {run_id}")

In [0]:
folder_listing = spark.sql(f"""
SELECT path, file, size, modification_time
FROM list_files('{DOT_FOLDER}', connection => '{GOOGLE_DRIVE_CONNECTION}')
ORDER BY path
""")
print(f"Total items found: {folder_listing.count()}")
display(folder_listing)

# Fill each year's exact Drive filename by matching file IDs
def _file_id(u):
    u = str(u)
    m = re.search(r"/file/d/([^/?&]+)", u) or re.search(r"[?&]id=([^&]+)", u)
    return m.group(1) if m else u

from urllib.parse import unquote

def _filename(path_value):
    """The Drive connector returns the name inside the path, e.g. '...?mimeType=...&filename=NAME.pdf'."""
    m = re.search(r"[?&]filename=([^&']+)", str(path_value))
    return unquote(m.group(1)) if m else None

id_to_name = {}
for r in folder_listing.select("path", "file").collect():
    file_val = r["file"]
    uri = None
    try:
        uri = file_val["uri"]          # struct/Row returned by list_files
    except Exception:
        uri = getattr(file_val, "uri", None)
    name = _filename(r["path"]) or _filename(uri) or _filename(file_val)
    if name:
        id_to_name[_file_id(str(r["path"]))] = name
missing = []
for year in years:
    name = id_to_name.get(_file_id(SOURCE_FILES[year]["url"]))
    if name:
        SOURCE_FILES[year]["file"] = name
    else:
        missing.append(year)
for year in years:
    print(f"  {year}: {SOURCE_FILES[year]['file']}")
assert not missing, f"These files were not found in DOT_FOLDER: {missing}. Check the folder URL and file links."

## Ingest Each Yearly PDF

For each selected year: read the PDF bytes from Drive, extract rows, run checks, and write to Bronze.

If a **critical check fails**, nothing is written for that year. The file is recorded as `NEEDS_REVIEW` in `file_manifest` and the reason goes to `ingestion_errors`.

In [0]:
for year in years:
    src = SOURCE_FILES[year]
    url, fname = src["url"], src["file"] or f"DS_05_{year}.pdf"
    print(f"\n{'='*60}\n{year} — {fname}")
    try:
        binary_df = (spark.read.format("binaryFile")
                     .option("databricks.connection", GOOGLE_DRIVE_CONNECTION)
                     .load(url))
        pdf_bytes = binary_df.collect()[0]["content"]

        rows, text_lines, info = parse_dot_pdf(pdf_bytes)
        errors, notes = check_extraction(rows, info)
        if info["report_year"] != year:
            errors.append(f"File says {info['report_year']}, expected {year}")

        if errors:
            msg = "; ".join(errors)
            print(f"✗ NEEDS_REVIEW: {msg}")
            log_ingestion_error(fname, url, tbl_rows(year), "EXTRACTION_CHECK_FAILED", msg)
            record_ingestion(fname, tbl_rows(year), 0, 0, "NEEDS_REVIEW", msg, url)
            continue

        common = {"source_year": year, "report_title": info["report_title"], "report_as_of": info["report_as_of"]}

        # 1) Table rows (all values as text, exactly as printed)
        row_cols = ["source_year", "report_title", "report_as_of", "page_number", "pdf_line_number", "indent_level",
                    "parent_label", "area_label", "foreign_travelers", "overseas_filipinos", "domestic_travelers", "total_travelers"]
        row_data = [tuple(None if (v := {**common, **r}.get(c)) is None else str(v) for c in row_cols) for r in rows]
        rows_df = add_bronze_metadata(
            spark.createDataFrame(row_data, schema=", ".join(f"{c} STRING" for c in row_cols)), fname, url)
        write_table(rows_df, tbl_rows(year))

        # 2) Non-table lines (titles, as-of date, footnotes)
        text_cols = ["source_year", "report_title", "report_as_of", "page_number", "pdf_line_number", "line_text"]
        text_data = [tuple(None if (v := {**common, **t}.get(c)) is None else str(v) for c in text_cols) for t in text_lines]
        text_df = add_bronze_metadata(
            spark.createDataFrame(text_data, schema=", ".join(f"{c} STRING" for c in text_cols)), fname, url)
        write_year_part(text_df, TBL_TEXT, year)

        # 3) Original PDF as binary
        pdf_df = add_bronze_metadata(
            binary_df.select(F.lit(year).alias("source_year"), F.col("path"),
                             F.col("length").alias("file_size"), F.col("content").alias("binary_content")),
            fname, url)
        write_year_part(pdf_df, TBL_PDF, year)

        check_note = (f"pages={info['pages']}; as_of={info['report_as_of']}; rows={len(rows)}; "
                      f"all row sums and GRAND TOTAL reconcile" + (f"; {' | '.join(notes)}" if notes else ""))
        for tbl, df_ in [(tbl_rows(year), rows_df), (TBL_TEXT, text_df), (TBL_PDF, pdf_df)]:
            n = spark.table(bronze_table(tbl)).filter(F.col("source_year") == year).count()
            record_ingestion(fname, tbl, n, len(df_.columns), "INGESTED", check_note, url)
        print(f"✓ INGESTED: {len(rows)} table rows, {len(text_lines)} text lines | {check_note}")

    except Exception as e:
        print(f"✗ FAILED: {year} — {e}")
        log_ingestion_error(fname, url, tbl_rows(year), "INGESTION_ERROR", str(e))
        record_ingestion(fname, tbl_rows(year), 0, 0, "FAILED", str(e), url)

## Validate Bronze Tables

Read back each year's tables and confirm metadata columns, a single source file, and row counts. Rerunning the same year should give the same counts (no duplicates).

In [0]:
checks = []
def _check(tbl, t, year):
    for col in METADATA_COLS:
        assert col in t.columns, f"Missing metadata column {col} in {tbl}"
    assert t.filter(F.col("_source_system") != SOURCE_SYSTEM).count() == 0
    assert t.filter(F.col("_source_dataset_id") != DATASET_ID).count() == 0
    assert t.filter(F.col("_source_path").isNull() | F.col("_ingested_at").isNull()).count() == 0
    assert t.filter(F.col("_source_sheet").isNotNull()).count() == 0, f"_source_sheet should be NULL for PDF in {tbl}"
    assert t.select("_source_file").distinct().count() == 1, f"More than one source file for {year} in {tbl}"

for year in years:
    # per-year data table
    tbl = tbl_rows(year)
    if spark.catalog.tableExists(bronze_table(tbl)):
        t = spark.table(bronze_table(tbl))
        assert t.filter(F.col("source_year") != year).count() == 0, f"Wrong year in {tbl}"
        _check(tbl, t, year)
        checks.append((year, tbl, t.count(), "ok"))
    else:
        checks.append((year, tbl, None, "not created"))
    # combined tables (this year's rows only)
    for tbl in [TBL_TEXT, TBL_PDF]:
        if spark.catalog.tableExists(bronze_table(tbl)):
            t = spark.table(bronze_table(tbl)).filter(F.col("source_year") == year)
            if t.count() == 0:
                checks.append((year, tbl, 0, "no rows for this year"))
                continue
            _check(tbl, t, year)
            checks.append((year, tbl, t.count(), "ok"))
        else:
            checks.append((year, tbl, None, "not created"))
display(spark.createDataFrame(checks, "year STRING, table STRING, row_count BIGINT, result STRING"))

## Ingestion Summary and Run Log Update

Updates this run's `ingestion_run_log` row from `STARTED` to `SUCCESS`, `PARTIAL_SUCCESS`, or `FAILED`.

In [0]:
manifest_df = spark.table(ingestion_table("file_manifest")).filter(F.col("run_id") == run_id)
manifest_df.createOrReplaceTempView("_run_manifest")
spark.sql("""
SELECT source_file,
  CASE WHEN SUM(CASE WHEN status = 'FAILED' THEN 1 ELSE 0 END) > 0 THEN 'FAILED'
       WHEN SUM(CASE WHEN status = 'INGESTED' THEN 1 ELSE 0 END) > 0 THEN 'INGESTED'
       ELSE 'SKIPPED' END AS file_status
FROM _run_manifest GROUP BY source_file
""").createOrReplaceTempView("_file_rollup")

files_discovered = spark.sql("SELECT COUNT(*) FROM _file_rollup").collect()[0][0]
files_ingested   = spark.sql("SELECT COUNT(*) FROM _file_rollup WHERE file_status='INGESTED'").collect()[0][0]
files_skipped    = spark.sql("SELECT COUNT(*) FROM _file_rollup WHERE file_status='SKIPPED'").collect()[0][0]
files_failed     = spark.sql("SELECT COUNT(*) FROM _file_rollup WHERE file_status='FAILED'").collect()[0][0]
tables_created   = spark.sql("SELECT COUNT(DISTINCT destination_table) FROM _run_manifest WHERE status='INGESTED'").collect()[0][0]
error_count      = spark.table(ingestion_table("ingestion_errors")).filter(F.col("run_id") == run_id).count()

if files_failed == 0 and files_skipped == 0 and error_count == 0:
    final_status = "SUCCESS"
elif files_ingested > 0:
    final_status = "PARTIAL_SUCCESS"
else:
    final_status = "FAILED"
final_notes = (f"DOT ingestion: {files_discovered} files processed, {files_ingested} ingested, "
               f"{files_skipped} needs review, {files_failed} failed, {tables_created} Bronze tables, {error_count} errors.")

spark.sql(f"""
UPDATE {ingestion_table('ingestion_run_log')}
SET status = :status, completed_at = current_timestamp(),
    files_discovered = :fd, files_ingested = :fi, files_skipped = :fs, files_failed = :ff,
    tables_created = :tc, notes = :notes
WHERE run_id = :run_id
""", args={"status": final_status, "fd": files_discovered, "fi": files_ingested, "fs": files_skipped,
           "ff": files_failed, "tc": tables_created, "notes": final_notes, "run_id": run_id})

print(f"Run {run_id}: {final_status}\n{final_notes}")
display(spark.table(ingestion_table("ingestion_run_log")).filter(F.col("run_id") == run_id))
display(manifest_df.orderBy("source_file", "destination_table"))
display(spark.table(ingestion_table("ingestion_errors")).filter(F.col("run_id") == run_id))